# Taco Bueno — Data Quality Checks
Read-only report for `workspace.default.taco_bueno_clean`. This notebook does not recreate or modify tables.

**Levels of detail:** item line and ticket (`store_number`, `order_date`, `check_number`). Repeated rows do not prove duplication: no validated unique line identifier is available.

`net_total` is the reconstructed amount after the ticket discount and before tax. Use it only when `net_total_status = reconciled`, once per ticket. Arithmetic reconciliation does not certify the POS accounting interpretation. Excluded tickets are explicitly reported.


In [0]:
from pyspark.sql import functions as F
from datetime import datetime
TABLE_NAME = "workspace.default.taco_bueno_clean"
KEYS = ["store_number", "order_date", "check_number"]
df = spark.table(TABLE_NAME)
required = KEYS + ["quantity", "menu_price", "item_discount", "total", "total_discount", "tax", "net_total", "net_total_status", "net_item_price", "price_per_unit", "discount_pct", "is_voided", "order_voided", "item_voided", "transaction_type", "is_incomplete_store", "source_file"]
missing = sorted(set(required) - set(df.columns))
assert not missing, f"Missing columns from the corrected PART1: {missing}"
print(TABLE_NAME, datetime.now().isoformat())
def n(condition):
    return F.sum(F.when(condition, 1).otherwise(0))


workspace.default.taco_bueno_clean 2026-10-08T00:36:30.445643


## 1. Coverage, nulls, and data types
Report actual coverage, including 2025 business dates contained in 2026 files. Null net_total values for exceptions are intentional.

In [0]:
display(df.agg(
    F.count("*").alias("rows"), F.countDistinct("store_number").alias("stores"),
    F.min("order_date").alias("first_date"), F.max("order_date").alias("last_date"),
    F.countDistinct("order_date").alias("observed_days"),
    n(F.col("is_incomplete_store") == 1).alias("rows_in_excluded_stores"),
    n(F.col("quantity") == 0).alias("zero_quantity"),
    n(F.col("quantity") < 0).alias("negative_quantity"),
    n(F.col("menu_item") == "Unknown Item").alias("unknown_item")
))
null_counts = df.agg(*[n(F.col(c).isNull()).alias(c) for c in required]).first().asDict()
display(spark.createDataFrame([(c, int(v or 0)) for c,v in null_counts.items()], "column_name string, null_rows long"))
expected = {"store_number":"string", "check_number":"string", "order_date":"date", "quantity":"int", "total":"double", "menu_price":"double", "net_total":"double", "net_total_status":"string"}
actual = {x.name:x.dataType.simpleString() for x in df.schema.fields}
display(spark.createDataFrame([(c, t, actual.get(c,"MISSING"), actual.get(c)==t) for c,t in expected.items()], "column_name string, expected string, actual string, matches boolean"))


rows,stores,first_date,last_date,observed_days,rows_in_excluded_stores,zero_quantity,negative_quantity,unknown_item
42253092,115,2025-12-31,2026-09-27,256,349755,74737,0,4902303


column_name,null_rows
store_number,0
order_date,0
check_number,0
quantity,0
menu_price,0
item_discount,0
total,0
total_discount,0
tax,0
net_total,536489


column_name,expected,actual,matches
store_number,string,string,true
check_number,string,string,true
order_date,date,date,true
quantity,int,int,true
total,double,double,true
menu_price,double,double,true
net_total,double,double,true
net_total_status,string,string,true


## 2. Tickets and consistency
Check that repeated ticket fields do not change across lines. `max` extracts an amount, but financial summaries use it only when consistency has also been validated.

In [0]:
tickets = df.groupBy(*KEYS).agg(
    F.count("*").alias("lines"), F.sum("quantity").alias("recorded_units"),
    F.sum(F.col("quantity")*F.col("menu_price")).alias("gross_amount"),
    F.max("total").alias("total"), F.max("total_discount").alias("ticket_discount"),
    F.max("tax").alias("ticket_tax"), F.max("net_total").alias("net_total"),
    F.max("net_total_status").alias("status"),
    F.countDistinct("net_total_status").alias("distinct_statuses"),
    F.countDistinct("net_total").alias("distinct_net_amounts"),
    n(F.col("net_total").isNull()).alias("null_net_amounts"),
    n(F.col("net_total_status").isNull()).alias("null_statuses"),
    F.countDistinct("total").alias("n_total"),
    F.countDistinct("total_discount").alias("distinct_discounts"),
    F.countDistinct("tax").alias("distinct_tax_amounts"),
    F.countDistinct("source_file").alias("distinct_source_files"),
    F.max("has_discount").alias("has_discount"),
    F.max("is_voided").alias("has_any_void"),
    F.max("is_incomplete_store").alias("incomplete_store"),
    F.max(F.when(
        F.col("quantity").isNull() | F.col("menu_price").isNull() |
        F.col("total").isNull() | F.col("total_discount").isNull() |
        F.col("tax").isNull() | (F.col("quantity") < 0) |
        (F.col("item_discount") < 0) | (F.col("total_discount") < 0),1
    ).otherwise(0)).alias("invalid_amounts")
)
key_ok = F.col("store_number").isNotNull() & F.col("order_date").isNotNull() & F.col("check_number").isNotNull()
consistent = key_ok & (F.col("n_total")==1) & (F.col("distinct_discounts")==1) & (F.col("distinct_tax_amounts")==1) & (F.col("invalid_amounts")==0)
display(tickets.agg(
    F.count("*").alias("tickets"), n(~key_ok).alias("groups_with_missing_keys"),
    n(~consistent).alias("tickets_with_invalid_amounts"),
    n(F.col("distinct_source_files")>1).alias("tickets_in_multiple_files"),
    F.avg("lines").alias("average_lines_not_units")
))


tickets,groups_with_missing_keys,tickets_with_invalid_amounts,tickets_in_multiple_files,average_lines_not_units
6066778,0,1,0,6.964667571485227


## 3. Independent net amount reconciliation
Reevaluate both observed Total conventions using the published lines. Do not subtract line discounts in addition to the ticket discount. Verify the stored status and net amount.

In [0]:
match_gross = F.abs(F.col("total")-F.col("gross_amount")) <= 0.02
match_paid = F.abs(F.col("total")-(F.col("gross_amount")-F.col("ticket_discount")+F.col("ticket_tax"))) <= 0.02
expected_status = (F.when(F.col("has_any_void")!=0,"review_void")
    .when(~consistent,"review_amounts")
    .when(F.col("gross_amount")-F.col("ticket_discount") < -0.02,"review_negative_net")
    .when(match_gross | match_paid,"reconciled")
    .otherwise("review_unreconciled"))
expected_net = F.when(expected_status == "reconciled", F.round(F.col("gross_amount")-F.col("ticket_discount"),2))
net_equal = ((F.col("net_total").isNull() & expected_net.isNull()) |
    F.coalesce(F.abs(F.col("net_total")-expected_net)<=0.005, F.lit(False)))
tickets = (tickets.withColumn("expected_status",expected_status)
    .withColumn("net_amount_correct",net_equal)
    .withColumn("status_correct",F.col("status").eqNullSafe(expected_status) & (F.col("distinct_statuses")==1) & (F.col("null_statuses")==0)))
valid_net = (F.col("status")=="reconciled") & F.col("status_correct") & F.col("net_amount_correct") & (F.col("distinct_net_amounts")==1) & (F.col("null_net_amounts")==0)
display(tickets.groupBy("incomplete_store","status").count().orderBy("incomplete_store","status"))
display(tickets.agg(
    n(~F.col("status_correct")).alias("tickets_with_incorrect_status"),
    n(~F.col("net_amount_correct")).alias("tickets_with_incorrect_net_formula"),
    n((F.col("status")=="reconciled") & ((F.col("distinct_net_amounts")!=1) | (F.col("null_net_amounts")>0))).alias("inconsistent_reconciled_net_amounts"),
    n((F.col("status")!="reconciled") & (F.col("null_net_amounts")!=F.col("lines"))).alias("exceptions_with_nonnull_net")
))


incomplete_store,status,count
0,reconciled,5948724
0,review_negative_net,1
0,review_unreconciled,2145
0,review_void,61325
1,reconciled,54091
1,review_amounts,1
1,review_negative_net,1
1,review_unreconciled,3
1,review_void,487


tickets_with_incorrect_status,tickets_with_incorrect_net_formula,inconsistent_reconciled_net_amounts,exceptions_with_nonnull_net
0,0,0,0


## 4. Line formulas and voids
Nulls are expected for percentages with a zero denominator and for unit prices without a positive quantity. Negative discounts are reported, not converted to positive values.

In [0]:
def different(actual, expected, tolerance=0.000001):
    return ~((F.col(actual).isNull() & expected.isNull()) | F.coalesce(F.abs(F.col(actual)-expected)<=tolerance,F.lit(False)))
line_net = F.col("quantity")*F.col("menu_price")-F.col("item_discount")
unit_net = F.when(F.col("quantity")>0,line_net/F.col("quantity"))
pct = F.when((F.col("quantity")>0)&(F.col("menu_price")>0),100*F.col("item_discount")/(F.col("quantity")*F.col("menu_price")))
void = F.when((F.coalesce(F.col("order_voided"),F.lit(0))!=0) | (F.coalesce(F.col("item_voided"),F.lit(0))!=0) | (F.lower(F.trim(F.col("transaction_type")))=="void"),1).otherwise(0)
display(df.agg(
    n(different("net_item_price",line_net)).alias("line_net_errors"),
    n(different("price_per_unit",unit_net)).alias("unit_price_errors"),
    n(different("discount_pct",pct)).alias("percentage_errors"),
    n(~F.col("is_voided").eqNullSafe(void)).alias("void_flag_errors"),
    n(F.col("discount_pct")<0).alias("negative_percentages"),
    n(F.col("discount_pct")>100.01).alias("percentages_over_100"),
    n(void==1).alias("voided_lines")
))
display(df.groupBy("is_incomplete_store","transaction_type","is_voided").count())


line_net_errors,unit_price_errors,percentage_errors,void_flag_errors,negative_percentages,percentages_over_100,voided_lines
0,0,0,0,3,0,157162


is_incomplete_store,transaction_type,is_voided,count
0,Regular Sale,0,41737409
1,Regular Sale,1,1000
0,Regular Sale,1,154509
1,Regular Sale,0,348182
0,Void,1,1607
1,Discount,0,527
1,Void,1,46
0,Discount,0,9812


## 5. Repeated items and date coverage
Repeated items within a ticket may be legitimate. This report does not instruct users to delete them. Days without records do not prove that a store was open or that sales are missing. Compare each store's observed date range without an arbitrary 200-day threshold.

In [0]:
repeated_items = (df.groupBy(*KEYS,"menu_item","begin_time")
    .count().filter(F.col("count")>1))
display(repeated_items.agg(F.count("*").alias("repeated_item_groups_not_confirmed_errors"),F.sum("count").alias("rows_in_repeated_groups")))
coverage = (df.groupBy("store_number").agg(
    F.min("order_date").alias("first_date"),F.max("order_date").alias("last_date"),
    F.countDistinct("order_date").alias("observed_days"),F.max("is_incomplete_store").alias("excluded_by_configuration"))
    .withColumn("days_in_range",F.datediff("last_date","first_date")+1)
    .withColumn("days_without_records_in_range",F.col("days_in_range")-F.col("observed_days")))
display(coverage.orderBy(F.desc("days_without_records_in_range")))


repeated_item_groups_not_confirmed_errors,rows_in_repeated_groups
8001433,24670994


store_number,first_date,last_date,observed_days,excluded_by_configuration,days_in_range,days_without_records_in_range
5385,2025-12-31,2026-09-27,251,0,271,20
9100,2025-12-31,2026-09-27,252,0,271,19
6143,2025-12-31,2026-09-27,252,0,271,19
9041,2025-12-31,2026-09-27,253,0,271,18
5744,2025-12-31,2026-09-27,253,0,271,18
9522,2025-12-31,2026-09-27,253,0,271,18
5653,2025-12-31,2026-09-27,253,0,271,18
9990,2025-12-31,2026-09-27,253,0,271,18
5026,2025-12-31,2026-09-27,253,0,271,18
7193,2025-12-31,2026-09-27,253,0,271,18


## 6. Team summary
Metrics are calculated per ticket after excluding stores flagged as incomplete. The net sum covers **only reconciled and verified tickets**, not all sales. Original Total amounts are not presented as net revenue.

In [0]:
scope = tickets.filter(F.col("incomplete_store")==0)
display(scope.agg(
    F.count("*").alias("tickets_in_scope"),
    n(valid_net).alias("tickets_with_validated_net"),
    (100*F.avg(F.when(valid_net,1.0).otherwise(0.0))).alias("net_coverage_pct"),
    F.round(F.sum(F.when(valid_net,F.col("net_total"))),2).alias("net_before_tax_validated_only"),
    F.round(F.avg(F.when(valid_net,F.col("net_total"))),2).alias("average_net_validated_only"),
    n(F.col("has_any_void")!=0).alias("tickets_with_any_void")
))
display(scope.filter(valid_net).groupBy("store_number").agg(
    F.count("*").alias("validated_tickets"),
    n(F.col("has_discount")==1).alias("tickets_with_discount"),
    F.round(F.sum("ticket_discount"),2).alias("ticket_discount_counted_once"),
    F.round(F.sum("net_total"),2).alias("net_before_tax")
).orderBy("store_number"))
print("Report complete. Review formula/status errors and document coverage; completing execution does not resolve exceptions.")


tickets_in_scope,tickets_with_validated_net,net_coverage_pct,net_before_tax_validated_only,average_net_validated_only,tickets_with_any_void
6012195,5948724,98.94429571895124,8.025659543E7,13.49,61325


store_number,validated_tickets,tickets_with_discount,ticket_discount_counted_once,net_before_tax
5026,48677,4352,8745.52,620694.44
5053,46262,3972,14430.41,534739.31
5094,61027,4683,18922.88,867952.62
5260,58646,6209,15767.34,763076.21
5261,27672,1301,9720.21,405838.32
5262,62794,3952,12102.58,863901.24
5265,45513,4243,11369.23,642557.2
5273,43121,1496,10941.59,621977.99
5355,45215,2371,7072.7,583720.97
5375,49166,3738,12722.47,675572.77


Report complete. Review formula/status errors and document coverage; completing execution does not resolve exceptions.


## Deliverables and interpretation
Save this executed notebook as `02_data_quality_checks.ipynb`, alongside the preparation notebook and README. Export reconciliation and summary tables if evidence files are needed.

- Zero formula errors confirms consistency with the implemented definition, not the absence of all source data errors.
- Tickets with voids are excluded from reconciled net amounts, even when they contain valid sales. Disclose this exclusion.
- Do not sum net_total or total_discount across item lines. Always group by all three ticket key columns.
- Do not rerun preparation: this notebook only reads the final table.


In [0]:
# Stop the pipeline if critical validation checks fail.
# Correctly classified source exceptions remain allowed.

ticket_checks = tickets.agg(
    n(~F.col("status_correct")).alias("incorrect_status"),
    n(~F.col("net_amount_correct")).alias("incorrect_net_formula"),
    n(
        (F.col("status") == "reconciled")
        & (
            (F.col("distinct_net_amounts") != 1)
            | (F.col("null_net_amounts") > 0)
        )
    ).alias("inconsistent_reconciled_net"),
    n(
        (F.col("status") != "reconciled")
        & (F.col("null_net_amounts") != F.col("lines"))
    ).alias("exceptions_with_nonnull_net")
).first().asDict()

line_checks = df.agg(
    F.count("*").alias("row_count"),
    n(different("net_item_price", line_net)).alias("line_net_errors"),
    n(different("price_per_unit", unit_net)).alias("unit_price_errors"),
    n(different("discount_pct", pct)).alias("percentage_errors"),
    n(~F.col("is_voided").eqNullSafe(void)).alias("void_flag_errors")
).first().asDict()

failures = {
    name: int(value)
    for name, value in ticket_checks.items()
    if value is not None and value > 0
}

for name, value in line_checks.items():
    if name != "row_count" and value is not None and value > 0:
        failures[name] = int(value)

if line_checks["row_count"] == 0:
    failures["empty_table"] = 1

for column in KEYS:
    if null_counts[column]:
        failures[f"missing_{column}"] = int(null_counts[column])

schema_types = {
    field.name: field.dataType.simpleString()
    for field in df.schema.fields
}
for column, expected_type in expected.items():
    if schema_types.get(column) != expected_type:
        failures[f"wrong_type_{column}"] = schema_types.get(column)

if failures:
    raise RuntimeError(f"DATA QUALITY GATE FAILED: {failures}")

print("DATA QUALITY GATE PASSED: downstream analyses may proceed.")